# Indexing

In [ ]:
from haystack import Pipeline, Document
from milvus_haystack import MilvusDocumentStore
from haystack.components.converters import PyPDFToDocument
from haystack.components.preprocessors import DocumentCleaner, DocumentSplitter
from haystack.components.embedders import SentenceTransformersDocumentEmbedder
from haystack.components.writers import DocumentWriter
import os
import glob

# Learning materials to tag
documents_dir = "/home/ryanwtsai/repos/ml_wcd/slides"

""" Connect to Milvus Lite DB; create new collection and drop old if exists """
document_store = MilvusDocumentStore(
    collection_name="MLE_Bootcamp_Prompt_Eng",
    collection_description="Prompt development using MLE bootcamp learning materials",
    connection_args={"uri": "./milvus.db"}, # Milvus Lite
    #connection_args={"uri": "http://localhost:19530"}, # Milvus standalone Docker service
    index_params={
        "metric_type": "L2",
        "index_type": "IVF_FLAT",
    },
    drop_old=True,
)

""" Connect to Milvus Lite DB; connect to existing collection """
# document_store = MilvusDocumentStore(
#     collection_name="MLE_Bootcamp_Prompt_Eng",
#     # collection_description="Prompt development using MLE bootcamp learning materials",
#     connection_args={"uri": "./milvus.db"}, # Milvus Lite
#     #connection_args={"uri": "http://localhost:19530"}, # Milvus standalone Docker service
#     # index_params={
#     #     "metric_type": "L2",
#     #     "index_type": "IVF_FLAT",
#     # },
#     drop_old=False,
# )

In [ ]:
pipe = Pipeline()

pipe.add_component("converter", PyPDFToDocument(extraction_mode="layout"))
pipe.add_component("cleaner", DocumentCleaner())
pipe.add_component("splitter", DocumentSplitter(split_by="word", split_length=100, split_overlap=20, split_threshold=50))
pipe.add_component("embedder", SentenceTransformersDocumentEmbedder())
pipe.add_component("writer", DocumentWriter(document_store=document_store))

pipe.connect("converter", "cleaner")
pipe.connect("cleaner", "splitter")
pipe.connect("splitter", "embedder")
pipe.connect("embedder", "writer")

In [5]:
file_names = glob.glob(f"{documents_dir}/**/*.pdf")
# for name in file_names:
#     print(name)

In [ ]:
results = pipe.run({"converter": {"sources": file_names}}, include_outputs_from={"converter", "cleaner", "splitter", "embedder"})

In [ ]:
print(len(results["converter"]["documents"]))
print("-"*10)
print(results["converter"]["documents"][0].content)

In [ ]:
print(len(results["cleaner"]["documents"]))
print("-"*10)
print(results["cleaner"]["documents"][0].content)

In [ ]:
print(len(results["splitter"]["documents"]))
print("-"*10)
print(results["splitter"]["documents"][0])

In [ ]:
print(len(results["embedder"]["documents"]))
print("-"*10)
print(results["embedder"]["documents"][0])

In [7]:
results["writer"]

{'documents_written': 996}

In [8]:
print(document_store.count_documents())
print("-"*10)
print(document_store.filter_documents()[0])

996
----------
Document(id=00553c23bd7483063c678f8881955977b18bbad2f3334da76ae231893c104d34, content: 'C C D D Reference: https://huggingface.co/blog/evaluating-mmlu-leaderboardDiﬀerent ways of evaluati...', meta: {'file_path': 'Lec8 LLM Performance Evaluation.pdf', 'page_number': 102, 'source_id': '588555c888a1a2502f6c3cf1ee581cd6373a5ce00ddaeef1595f0d347915fc72', 'split_id': 60, 'split_idx_start': 37178}, embedding: vector of size 768)


# Prompt Dev

In [41]:
# meta_filter = {"field": "meta.file_path", "operator": "==", "value": "Lec1 Machine Learning Review.pdf"} # doc 31 - traditional ML
meta_filter = {"field": "meta.file_path", "operator": "==", "value": "Lec5 Neural Networks Basics.pdf"} # doc 10 - deep learning
# meta_filter = {"field": "meta.file_path", "operator": "==", "value": "Lec7 Introduction to CNN.pdf"} # doc 2 - CV
# meta_filter = {"field": "meta.file_path", "operator": "==", "value": "Lec4 BERT.pdf"} # doc 1 - NLP
# meta_filter = {"field": "meta.file_path", "operator": "==", "value": "Lec3 Docker Basics.pdf"} # doc 2 - MLOps/DevOps
docs = document_store.filter_documents(meta_filter)
for idx, doc in enumerate(docs):
    print(f"Doc {idx}:")
    print(doc.content)
    print("-"*10)

Doc 0:
‘s ? Machine Learning Engineering Program | www.weclouddata.comCost Function (Least Squares) Linear Regression Choose so that is close to
for training examples Machine Learning Engineering Program | www.weclouddata.comCost Function (Least Squares) Linear Regression Hypothesis: Parameters: Choose so that is close to
for training examples Machine Learning Engineering Program | www.weclouddata.comCost Function (Least Squares) Linear Regression Hypothesis: Parameters: Cost Function: Choose so that is close to Goal:
for training examples Machine Learning Engineering Program | www.weclouddata.comCost Function (Least Squares)
Linear Regression Machine Learning Engineering Program | www.weclouddata.comCost Function (Least Squares)
Linear Regression After exhaustively trying different values of we get a contour plot which captures the 
----------
Doc 1:
Regression - Cost Function Logistic Regression •Cross-entropy loss, or log loss, measures the performance of a classification model

In [42]:
docs = []
meta_filter = {"field": "meta.file_path", "operator": "==", "value": "Lec1 Machine Learning Review.pdf"} # doc 31 - traditional ML
docs.append(document_store.filter_documents(meta_filter)[31])
meta_filter = {"field": "meta.file_path", "operator": "==", "value": "Lec5 Neural Networks Basics.pdf"} # doc 10 - deep learning
docs.append(document_store.filter_documents(meta_filter)[15])
meta_filter = {"field": "meta.file_path", "operator": "==", "value": "Lec7 Introduction to CNN.pdf"} # doc 2 - CV
docs.append(document_store.filter_documents(meta_filter)[2])
meta_filter = {"field": "meta.file_path", "operator": "==", "value": "Lec4 BERT.pdf"} # doc 1 - NLP
docs.append(document_store.filter_documents(meta_filter)[1])
meta_filter = {"field": "meta.file_path", "operator": "==", "value": "Lec3 Docker Basics.pdf"} # doc 2 - MLOps/DevOps
docs.append(document_store.filter_documents(meta_filter)[2])
for doc in docs:
    print(doc.content)
    print("-"*10)

IEEE, 2007.Support vectors
Support vectors: Vectors that at the boundary of the margin based on data
points that lie closest to the decision boundary. Support vector Support vector feature 1 (x1) Machine Learning Engineering Program | www.weclouddata.comSupervised Learning
Ensemble ModelsEnsemble learning (concept) Marie Jean Antoine Nicolas de Caritat
(French mathematician; 1743–1794) Condorcetʼs jury theorem in 1785:
● Each voter has a probability p > .5 of being correct (better than a random guess) ○ adding more voters increases the probability of making the correct decision Machine Learning Engineering Program | www.weclouddata.comEnsemble learning methods Majority Averaging Weighted Average Stacking Bagging Boosting Python Fundamentals | Data Science 
----------
PyTorch DL Frameworks It’s a Python-based scientific computing package targeted at two sets of audiences: ● A replacement for NumPy to use the power of GPUs ● A deep learning research platform that provides maximum fle

In [32]:
import openai
import os

openai.api_key = os.environ["OPENAI_API_KEY"]
client = openai.OpenAI()

def get_completion(prompt, model="gpt-3.5-turbo"):
    messages = [{"role": "user", "content": prompt}]
    response = client.chat.completions.create(
        model=model,
        messages=messages,
        temperature=0
    )
    return response.choices[0].message.content

In [ ]:
prompt = f"""
You will be provided a chunk of text or code, delimited in triple backticks, related to some subfield of machine learning engineering.

You will identify the subfield. The possible subfields are in this list: [traditional machine learning, deep learning, computer vision, natural language processing, MLOps].

If you think the text belongs to a subfield not in the list, give your best guess.

Respond with only the subfield and no other text.

Chunk of text or code to identify:
```{doc}```
"""

prompt = f"""
    You will be provided a chunk of text or code, delimited in triple backticks, related to some subfield of machine learning engineering.

    You will identify the subfield. The possible subfields are in this list: [`traditional machine learning`, `deep learning`, `computer vision`, `natural language processing`, `MLOps`].

    If multiple subfields apply, then you will choose the most specific subfield.
    `computer vision` and `natural language processing` are more specific than `deep learning`, which is more specific than `traditional machine learning`.

    Convolutional neural networks should be tagged as `computer vision`.

    If you think the text belongs to a subfield not in the list, give your best guess.

    Respond with only the subfield and no other text.

    Chunk of text or code to identify:
    ```{doc.content}```
    """

prompt = f"""
    You will be provided a chunk of text or code, delimited in triple backticks, related to some subfield of machine learning engineering.

    You will identify the subfield. The possible subfields are in this list: [`traditional machine learning`, `computer vision`, `natural language processing`, `MLOps`].

    If multiple subfields apply, then you will choose the most specific subfield.

    If you think the text belongs to a subfield not in the list, give your best guess.

    Respond with only the subfield and no other text.

    Chunk of text or code to identify:
    ```{doc.content}```
    """

prompt = f"""
    You will be provided a chunk of text or code, delimited in triple backticks, related to some subfield of machine learning engineering.

    Your task is to identify the most related subfield. The subfields are in this list:
    [`traditional machine learning`, `deep learning`, `computer vision`, `natural language processing`, `MLOps`]

    Here are some examples of topics for each subfield:
    1. `traditional machine learning`: linear regression, logistic regression, gradient descent, decision trees, SVM
    2. `deep learning`: dense neural networks, activation functions
    3. `computer vision`: convolutional neural networks, image classification, object detection, segmentation
    4. `natural language processing`: transformers, LLMs, RAG, AI agents
    5. `MLOps`: model deployment, containerization, cloud, model monitoring, pipelines

    Respond with only the subfield and no other text.

    Chunk of text or code to identify:
    ```{doc.content}```
    """

In [43]:
for doc in docs:
    prompt = f"""
    You will be provided a chunk of text or code, delimited in triple backticks, related to some subfield of machine learning engineering.

    Your task is to identify the most related subfield. The subfields are in this list:
    [`traditional machine learning`, `deep learning`, `computer vision`, `natural language processing`, `MLOps`]

    Here are some examples of topics for each subfield:
    1. `traditional machine learning`: linear regression, logistic regression, gradient descent, decision trees, SVM
    2. `deep learning`: dense neural networks, activation functions
    3. `computer vision`: convolutional neural networks, image classification, object detection, segmentation
    4. `natural language processing`: transformers, LLMs, RAG, AI agents
    5. `MLOps`: model deployment, containerization, cloud, model monitoring, pipelines

    Respond with only the subfield and no other text.

    Chunk of text or code to identify:
    ```{doc.content}```
    """
    tag = get_completion(prompt)
    print(f"Tag: {tag}")
    print(doc.content)
    print("-"*10)

Tag: `traditional machine learning`
IEEE, 2007.Support vectors
Support vectors: Vectors that at the boundary of the margin based on data
points that lie closest to the decision boundary. Support vector Support vector feature 1 (x1) Machine Learning Engineering Program | www.weclouddata.comSupervised Learning
Ensemble ModelsEnsemble learning (concept) Marie Jean Antoine Nicolas de Caritat
(French mathematician; 1743–1794) Condorcetʼs jury theorem in 1785:
● Each voter has a probability p > .5 of being correct (better than a random guess) ○ adding more voters increases the probability of making the correct decision Machine Learning Engineering Program | www.weclouddata.comEnsemble learning methods Majority Averaging Weighted Average Stacking Bagging Boosting Python Fundamentals | Data Science 
----------
Tag: `deep learning`
PyTorch DL Frameworks It’s a Python-based scientific computing package targeted at two sets of audiences: ● A replacement for NumPy to use the power of GPUs ● A 